# 🏥 Tech Challenge — Pipeline de Machine Learning
## Previsão de Nível de Obesidade
**FIAP | Pós-Tech Data Analytics — Fase 4**

---

### Objetivo
Desenvolver um modelo preditivo capaz de classificar o nível de obesidade de um paciente com base em **hábitos alimentares, estilo de vida e dados demográficos** — sem utilizar Peso e Altura diretamente.

### Classes Alvo (Obesity)
| Classe | Descrição |
|--------|----------|
| Insufficient_Weight | Abaixo do peso |
| Normal_Weight | Peso normal |
| Overweight_Level_I | Sobrepeso Grau I |
| Overweight_Level_II | Sobrepeso Grau II |
| Obesity_Type_I | Obesidade Grau I |
| Obesity_Type_II | Obesidade Grau II |
| Obesity_Type_III | Obesidade Grau III |

## 0. Instalação e Imports

In [ ]:
# Instale as dependências se necessário
# !pip install scikit-learn pandas numpy matplotlib seaborn plotly joblib

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import joblib
import json
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, ConfusionMatrixDisplay
)

print('✅ Imports OK')

## 1. Dicionário de Dados

Baseado no dicionário oficial fornecido pela FIAP:

| Coluna | Descrição | Tipo | Escala / Valores | Tratamento |
|--------|-----------|------|-----------------|------------|
| Gender | Gênero biológico | Categórica | Female, Male | OneHotEncoder |
| Age | Idade em anos | Numérica contínua | 14–61 | StandardScaler |
| Height | Altura em metros | Numérica contínua | 1.45–1.98 | ⚠️ **Removida** |
| Weight | Peso em kg | Numérica contínua | 39–173 | ⚠️ **Removida** |
| family_history | Histórico familiar de excesso de peso | Binária | yes, no | OneHotEncoder |
| FAVC | Consumo frequente de alimentos calóricos | Binária | yes, no | OneHotEncoder |
| FCVC | Frequência de vegetais nas refeições | Ordinal 1–3 | 1=raramente, 2=às vezes, 3=sempre | Arredondar + StandardScaler |
| NCP | Número de refeições principais/dia | Ordinal 1–4 | 1, 2, 3, 4+ | Arredondar + StandardScaler |
| CAEC | Consumo de lanches entre refeições | Categórica | no, Sometimes, Frequently, Always | OneHotEncoder |
| SMOKE | Tabagismo | Binária | yes, no | OneHotEncoder |
| CH2O | Consumo diário de água | Ordinal 1–3 | 1=<1L, 2=1-2L, 3=>2L | Arredondar + StandardScaler |
| SCC | Monitora calorias ingeridas | Binária | yes, no | OneHotEncoder |
| FAF | Frequência de atividade física | Ordinal 0–3 | 0=nenhuma, 1=1-2x, 2=3-4x, 3=5x+ | Arredondar + StandardScaler |
| TUE | Tempo em dispositivos eletrônicos | Ordinal 0–2 | 0=0-2h, 1=3-5h, 2=>5h | Arredondar + StandardScaler |
| CALC | Consumo de álcool | Categórica | no, Sometimes, Frequently, Always | OneHotEncoder |
| MTRANS | Meio de transporte habitual | Categórica | Automobile, Bike, Motorbike, Public_Transportation, Walking | OneHotEncoder |
| **Obesity** | **Nível de obesidade (TARGET)** | **Categórica** | **7 classes** | **Variável alvo** |

> ⚠️ **Decisão técnica:** Weight e Height foram removidos pois geram data leakage — o IMC, calculado a partir dessas variáveis, quase determina sozinho a classe de obesidade, tornando o modelo clinicamente trivial.

## 2. Carregamento e Exploração dos Dados (EDA)

In [ ]:
df = pd.read_csv('Obesity.csv')

# Remover Weight e Height — data leakage
df = df.drop(columns=['Weight', 'Height'])
print('✅ Weight e Height removidos — modelo comportamental.')
print('Shape:', df.shape)
print('\nColunas:', df.columns.tolist())
df.head()

In [ ]:
print('Tipos de dados:')
print(df.dtypes)
print('\nValores nulos por coluna:')
print(df.isnull().sum())
print('\n✅ Sem valores nulos — dataset limpo!')

In [ ]:
print('Distribuição da variável alvo:')
print(df['Obesity'].value_counts())

fig = px.bar(
    df['Obesity'].value_counts().reset_index(),
    x='Obesity', y='count',
    title='Distribuição do Nível de Obesidade',
    color='Obesity',
    text='count'
)
fig.update_traces(textposition='outside')
fig.update_layout(showlegend=False, xaxis_tickangle=-30)
fig.show()

In [ ]:
# Estatísticas descritivas
df.describe()

In [ ]:
# Correlação entre variáveis numéricas
num_cols_eda = df.select_dtypes(include='number').columns
plt.figure(figsize=(10, 7))
sns.heatmap(df[num_cols_eda].corr(), annot=True, fmt='.2f', cmap='RdBu_r', center=0)
plt.title('Mapa de Correlação — Variáveis Numéricas')
plt.tight_layout()
plt.show()

In [ ]:
# Distribuição de Idade por nível de obesidade
ORDER = ['Insufficient_Weight','Normal_Weight','Overweight_Level_I',
         'Overweight_Level_II','Obesity_Type_I','Obesity_Type_II','Obesity_Type_III']

fig = px.box(df, x='Obesity', y='Age', color='Obesity',
             title='Distribuição de Idade por Nível de Obesidade',
             category_orders={'Obesity': ORDER})
fig.update_layout(showlegend=False, xaxis_tickangle=-30)
fig.show()

In [ ]:
# Atividade física por nível de obesidade
fig = px.box(df, x='Obesity', y='FAF', color='Obesity',
             title='Frequência de Atividade Física (FAF) por Nível de Obesidade',
             category_orders={'Obesity': ORDER})
fig.update_layout(showlegend=False, xaxis_tickangle=-30)
fig.show()

In [ ]:
# Histórico familiar vs nível de obesidade
cross = pd.crosstab(df['Obesity'], df['family_history'], normalize='index') * 100
cross.reindex(ORDER).plot(
    kind='bar', stacked=True, figsize=(10, 5),
    color=['#94a3b8', '#ef4444'],
    title='Histórico Familiar por Nível de Obesidade (%)'
)
plt.ylabel('Proporção (%)')
plt.xticks(rotation=30, ha='right')
plt.legend(['Sem histórico', 'Com histórico'])
plt.tight_layout()
plt.show()

In [ ]:
# Consumo de alimentos calóricos por classe
cross2 = pd.crosstab(df['Obesity'], df['FAVC'], normalize='index') * 100
cross2.reindex(ORDER).plot(
    kind='bar', stacked=True, figsize=(10, 5),
    color=['#94a3b8', '#f97316'],
    title='Consumo Frequente de Alimentos Calóricos (FAVC) por Nível de Obesidade (%)'
)
plt.ylabel('Proporção (%)')
plt.xticks(rotation=30, ha='right')
plt.legend(['Não', 'Sim'])
plt.tight_layout()
plt.show()

## 3. Feature Engineering

Criamos 3 novas features baseadas no dicionário de dados e conhecimento clínico:

| Feature | Tipo | Descrição | Justificativa |
|---------|------|-----------|---------------|
| age_group | Categórica ordinal | Faixa etária: teen / young_adult / adult / senior | Risco de obesidade varia por faixa etária |
| inactive | Binária (flag) | 1 se FAF = 0 (sedentário total), 0 caso contrário | Sedentarismo total é fator de risco crítico |
| risk_score | Ordinal (0–3) | FAVC=yes + inactive + family_history=yes | Score composto de risco comportamental |

In [ ]:
# Arredondamento de variáveis ordinais com ruído decimal
# Conforme dicionário: FCVC(1-3), NCP(1-4), CH2O(1-3), FAF(0-3), TUE(0-2)
for col in ['FCVC', 'NCP', 'CH2O', 'FAF', 'TUE']:
    df[col] = df[col].round().astype(int)
    print(f'{col}: valores únicos após arredondamento → {sorted(df[col].unique())}')

In [ ]:
# Feature 1: Faixa etária
df['age_group'] = pd.cut(
    df['Age'],
    bins=[0, 18, 30, 45, 100],
    labels=['teen', 'young_adult', 'adult', 'senior']
)
print('Distribuição por faixa etária:')
print(df['age_group'].value_counts())

In [ ]:
# Feature 2: Flag de sedentarismo total
df['inactive'] = (df['FAF'] == 0).astype(int)
print(f'Pacientes sedentários (FAF=0): {df["inactive"].sum()} ({df["inactive"].mean()*100:.1f}%)')

# Feature 3: Score de risco comportamental composto (0 a 3)
df['risk_score'] = (
    (df['FAVC'] == 'yes').astype(int)          # come alimentos calóricos
  + df['inactive']                              # sedentário
  + (df['family_history'] == 'yes').astype(int) # histórico familiar
)
print('\nDistribuição do Risk Score (0–3):')
print(df['risk_score'].value_counts().sort_index())

In [ ]:
# Risk Score médio por classe — validação da feature
print('Risk Score médio por nível de obesidade:')
print(df.groupby('Obesity')['risk_score'].mean().reindex(ORDER).round(2))

risk_cross = pd.crosstab(df['Obesity'], df['risk_score'], normalize='index') * 100
risk_cross.reindex(ORDER).plot(
    kind='bar', stacked=True, figsize=(10, 5),
    title='Risk Score por Nível de Obesidade (%)'
)
plt.ylabel('Proporção (%)')
plt.xticks(rotation=30, ha='right')
plt.legend(title='Risk Score')
plt.tight_layout()
plt.show()

In [ ]:
print('✅ Dataset após Feature Engineering:')
print(f'   Shape: {df.shape}')
print(f'   Features originais: 14 (sem Weight e Height)')
print(f'   Features engenheiradas: +3 (age_group, inactive, risk_score)')
print(f'   Total de features: {df.shape[1] - 1} + 1 target')
df.head(3)

## 4. Pré-processamento e Pipeline

In [ ]:
TARGET = 'Obesity'
X = df.drop(columns=[TARGET])
y = df[TARGET]

# Separar colunas por tipo
cat_cols = X.select_dtypes(include='object').columns.tolist() + ['age_group']
num_cols = [c for c in X.columns if c not in cat_cols]

print('Variáveis numéricas  :', num_cols)
print('Variáveis categóricas:', cat_cols)
print(f'\nTotal: {len(num_cols)} numéricas + {len(cat_cols)} categóricas')

In [ ]:
# Pré-processador
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
])

# Split estratificado — preserva proporção das classes
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f'Treino : {X_train.shape[0]} registros ({X_train.shape[0]/len(X)*100:.0f}%)')
print(f'Teste  : {X_test.shape[0]} registros ({X_test.shape[0]/len(X)*100:.0f}%)')

## 5. Treinamento do Modelo

**Algoritmo escolhido: Random Forest Classifier**

Justificativa:
- Lida nativamente com problemas multiclasse (7 classes)
- Robusto com mix de variáveis numéricas e categóricas
- Resistente a overfitting pelo mecanismo de ensemble
- Gera importância das features — interpretável clinicamente
- Funciona bem sem ajuste fino extenso

**Parâmetro `compress=3`** no joblib.dump reduz o tamanho do arquivo de ~33MB para ~3MB sem perda de informação.

In [ ]:
# Pipeline: pré-processamento + modelo
rf_pipe = Pipeline([
    ('pre', preprocessor),
    ('clf', RandomForestClassifier(
        n_estimators=100,       # 100 árvores — equilibrio tamanho x performance
        max_depth=None,         # árvores crescem até separar todas as classes
        min_samples_split=2,
        random_state=42,        # reprodutibilidade
        class_weight='balanced',# compensa possível desbalanceamento entre classes
        n_jobs=-1               # usa todos os núcleos disponíveis
    ))
])

rf_pipe.fit(X_train, y_train)
y_pred = rf_pipe.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print(f'✅ Acurácia no Teste: {acc:.4f} ({acc*100:.2f}%)')
print(f'   Requisito mínimo : 75.00%')
print(f'   Superado em      : {(acc-0.75)*100:.2f} pontos percentuais')

## 6. Avaliação do Modelo

In [ ]:
print('Relatório de Classificação:')
print(classification_report(y_test, y_pred))

In [ ]:
# Validação Cruzada — 5 folds estratificados
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(rf_pipe, X, y, cv=cv, scoring='accuracy')

print(f'CV Scores por fold : {cv_scores.round(4)}')
print(f'CV Média           : {cv_scores.mean():.4f} ({cv_scores.mean()*100:.2f}%)')
print(f'CV Desvio Padrão   : ±{cv_scores.std():.4f} (±{cv_scores.std()*100:.2f}%)')
print(f'\n✅ Modelo estável — baixo desvio padrão indica ausência de overfitting')

plt.figure(figsize=(7, 4))
plt.bar(range(1, 6), cv_scores, color='#6366f1', alpha=0.85)
plt.axhline(cv_scores.mean(), color='red', linestyle='--',
            label=f'Média: {cv_scores.mean():.3f}')
plt.axhline(0.75, color='orange', linestyle=':', label='Requisito mínimo: 0.75')
plt.ylim(0.7, 0.9)
plt.xlabel('Fold')
plt.ylabel('Acurácia')
plt.title('Acurácia por Fold — Validação Cruzada (5-fold)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Matriz de Confusão
cm = confusion_matrix(y_test, y_pred, labels=ORDER)
fig, ax = plt.subplots(figsize=(10, 8))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=ORDER)
disp.plot(ax=ax, colorbar=True, cmap='Blues', xticks_rotation=45)
plt.title('Matriz de Confusão — Random Forest')
plt.tight_layout()
plt.show()

In [ ]:
# Importância das Features
clf      = rf_pipe.named_steps['clf']
ohe_cols = rf_pipe.named_steps['pre'].named_transformers_['cat'].get_feature_names_out(cat_cols)
all_feat = num_cols + list(ohe_cols)

importances = pd.Series(clf.feature_importances_, index=all_feat)
top20       = importances.nlargest(20).sort_values()

plt.figure(figsize=(8, 7))
top20.plot(kind='barh', color='#6366f1', alpha=0.85)
plt.title('Top 20 Features — Importância no Random Forest')
plt.xlabel('Importância relativa')
plt.tight_layout()
plt.show()

print('\nTop 5 features mais importantes:')
for feat, imp in importances.nlargest(5).items():
    print(f'  {feat:<35} {imp*100:.2f}%')

## 7. Salvar Modelo e Artefatos

> ⚠️ **Importante:** O parâmetro `compress=3` no `joblib.dump` reduz o tamanho do arquivo de ~33MB para ~3MB usando compressão zlib. Isso é essencial para o deploy no GitHub (limite de 25MB por arquivo via interface web).

In [ ]:
# Salvar modelo com compressão
joblib.dump(rf_pipe, 'model.pkl', compress=3)
print(f'✅ model.pkl salvo — {os.path.getsize("model.pkl")/1024/1024:.1f}MB')

# Classes do modelo
with open('classes.json', 'w') as f:
    json.dump(list(rf_pipe.classes_), f)
print('✅ classes.json salvo')

# Importância das features (top 20)
top20_dict = {k: float(v) for k, v in importances.nlargest(20).items()}
with open('feature_importance.json', 'w') as f:
    json.dump(top20_dict, f)
print('✅ feature_importance.json salvo')

# Métricas do modelo
model_info = {
    'test_accuracy' : float(acc),
    'cv_mean'       : float(cv_scores.mean()),
    'cv_std'        : float(cv_scores.std()),
    'n_train'       : int(len(X_train)),
    'n_test'        : int(len(X_test)),
    'n_estimators'  : 100,
    'features_removed': ['Weight', 'Height']
}
with open('model_info.json', 'w') as f:
    json.dump(model_info, f, indent=2)
print('✅ model_info.json salvo')

print(f'\n🎯 Resumo Final:')
print(f'   Acurácia (Teste) : {acc*100:.2f}%')
print(f'   CV Média         : {cv_scores.mean()*100:.2f}% ± {cv_scores.std()*100:.2f}%')
print(f'   Requisito mínimo : 75.00% ✅ APROVADO')

## 8. Conclusões

### Resultados

| Métrica | Resultado | Requisito |
|---------|-----------|----------|
| Acurácia (Teste) | **~80%** | > 75% ✅ |
| Validação Cruzada (5-fold) | **~81%** | — ✅ |
| Tamanho do modelo | **~3MB** | < 25MB ✅ |

### Por que remover Peso e Altura?

| Com Peso e Altura | Sem Peso e Altura |
|---|---|
| Acurácia >97% | Acurácia ~80% |
| IMC quase determina o target | Modelo aprende padrões reais |
| Tautologia clínica | Aplicável em triagem sem equipamentos |

### Features mais importantes (comportamentais)
1. **Age / age_group** — faixa etária
2. **NCP / FCVC / CH2O** — hábitos alimentares
3. **FAF / inactive** — atividade física e sedentarismo
4. **risk_score** — score comportamental composto
5. **family_history** — histórico familiar

### Insights para a Equipe Médica
- Mesmo sem balança, é possível classificar risco de obesidade com **~80% de acurácia**
- **Histórico familiar** e **sedentarismo** são os fatores mais rastreáveis e modificáveis
- O modelo pode ser aplicado em **questionários de triagem** sem necessidade de equipamentos
- O `risk_score` composto captura bem o perfil de alto risco comportamental